# Advection CNO Evaluation
This notebook evaluates standard CNO and Late-Fusion CNO on the advection setup, summarizes in-domain and out-domain accuracy, and visualizes the best-run predictions with heatmaps.

First we set the working directory and load libraries.

In [ ]:
from pathlib import Path
import os
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "src").exists() is False:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.utils.configs.training_config import load_config, build_run_plan
import src.utils.evaluation.evaluation as ev

importlib.reload(ev)

We compute for each trained model the in-domain and out-domain RMSE.

In [ ]:
CONFIG_PATH = Path("configs/training/advection_cno.yaml")
cfg = load_config(str(CONFIG_PATH))
runs = build_run_plan(cfg)

# Keep only the two model families requested
runs = [r for r in runs if r.get("model") in {"cno", "late_fusion"}]
print(f"Total planned runs (cno + late_fusion): {len(runs)}")

# Evaluate validation and test (ID + OD) for each run
val_rows = []
test_rows = []
failed_runs = []

for i, run_cfg in enumerate(runs, start=1):
    try:
        val_rows.append(ev.evaluate_run_validation(run_cfg, use_best=True))
        test_rows.extend(ev.evaluate_run_test(run_cfg, domains=("id", "od"), use_best=True, batch_size_test=500))
        print(f"[{i}/{len(runs)}] OK: {run_cfg['name']}")
    except Exception as e:
        failed_runs.append({"run_name": run_cfg["name"], "model": run_cfg["model"], "error": str(e)})
        print(f"[{i}/{len(runs)}] FAILED: {run_cfg['name']} -> {e}")

val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)
fail_df = pd.DataFrame(failed_runs)

if not fail_df.empty:
    display(fail_df)

if test_df.empty:
    raise RuntimeError("No test evaluations were produced.")

# Summary table: CNO vs Late-Fusion CNO, both ID and OD
summary = (
    test_df.groupby(["model", "domain"], as_index=False)
    .agg(
        rmse_mean=("test_rmse", "mean"),
        rmse_std=("test_rmse", "std"),
        n=("test_rmse", "count"),
    )
    .sort_values(["model", "domain"])
)
summary["rmse_std"] = summary["rmse_std"].fillna(0.0)
summary["rmse_pm_std"] = summary.apply(lambda r: f"{r['rmse_mean']:.3e} ± {r['rmse_std']:.3e}", axis=1)

display(summary)

table = (
    summary.pivot(index="model", columns="domain", values="rmse_pm_std")
    .rename(columns={"id": "In-domain", "od": "Out-domain"})
    .reset_index()
)

table["model"] = table["model"].replace({"cno": "CNO", "late_fusion": "Late-Fusion CNO"})
display(table)

# Save table
out_dir = Path("outputs/advection_cno")
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "cno_vs_latefusion_rmse_table.csv"
summary_path = out_dir / "cno_vs_latefusion_rmse_summary_long.csv"
table.to_csv(table_path, index=False)
summary.to_csv(summary_path, index=False)
print(f"Saved table: {table_path}")
print(f"Saved summary: {summary_path}")

Create visualisation of both CNO and Late-fusion CNO

In [ ]:
import src.utils.visualisations.visualise_results as vis_paper
importlib.reload(vis_paper)

if val_df.empty:
    raise RuntimeError("Validation results are empty; cannot select best runs for heatmap.")

# Choose best validation run per model
best_idx = val_df.groupby("model")["val_error"].idxmin()
best_val = val_df.loc[best_idx].reset_index(drop=True)

run_lookup = {r["name"]: r for r in runs}
best_runs_by_model = {
    row["model"]: run_lookup[row["run_name"]]
    for _, row in best_val.iterrows()
    if row["run_name"] in run_lookup
}

required_models = ["cno", "late_fusion"]
for m in required_models:
    if m not in best_runs_by_model:
        raise RuntimeError(f"Missing best run for model '{m}'. Available: {list(best_runs_by_model.keys())}")

bundle = ev.collect_best_run_predictions(
    best_runs_by_model=best_runs_by_model,
    plot_id_idx=2,
    plot_od_idx=3,
    batch_size_test=10,
    use_best=True,
)

# Dedicated 2-model heatmap function: GT + CNO + Late Fusion, with error maps and original colorbar style
fig = vis_paper.plot_id_od_heatmaps_two_models(
    bundle=bundle,
    font_size=9,
    model_order=["cno", "late_fusion"],
    panel_labels=("A", "B"),
)

fig_path_pdf = Path("outputs/advection_cno/cno_vs_latefusion_heatmap_id_od.pdf")
fig.savefig(fig_path_pdf, bbox_inches="tight")
plt.show()
print(f"Saved heatmap: {fig_path_pdf}")

Create latex table of performance.

In [ ]:
# LaTeX table (same style as paper table, but for current CNO results)
import re

if summary.empty:
    raise RuntimeError("`summary` is empty. Run the evaluation cell first.")

def _sci_no_zero_exp(x: float) -> str:
    s = f"{float(x):.2e}"
    # 4.72e-01 -> 4.72e-1
    return re.sub(r"e([+-])0*(\d+)", r"e\1\2", s)

def _pm(mean: float, std: float) -> str:
    if pd.isna(std):
        std = 0.0
    return f"{_sci_no_zero_exp(mean)} $\\pm$ {_sci_no_zero_exp(std)}"

# Build wide table for ID/OD
s = summary.copy()
s["model"] = s["model"].replace({"cno": "CNO", "late_fusion": "Late Fusion CNO"})

wide = s.pivot(index="model", columns="domain", values=["rmse_mean", "rmse_std"])

# Determine best model per domain (lowest mean RMSE)
best_id_model = wide[("rmse_mean", "id")].idxmin() if ("rmse_mean", "id") in wide.columns else None
best_od_model = wide[("rmse_mean", "od")].idxmin() if ("rmse_mean", "od") in wide.columns else None

model_order = [m for m in ["CNO", "Late Fusion CNO"] if m in wide.index] + [m for m in wide.index if m not in ["CNO", "Late Fusion CNO"]]

lines = []
lines.append(r"\begin{tabular}{lcc}")
lines.append(r"\toprule")
lines.append(r"Model & In-domain & Out-domain \\")
lines.append(r"\midrule")
lines.append(r"\multicolumn{3}{l}{\textbf{1D Advection (CNO Study)}} \\")
lines.append(r"\midrule")

for model in model_order:
    id_txt = "-"
    od_txt = "-"

    if ("rmse_mean", "id") in wide.columns:
        id_txt = _pm(wide.loc[model, ("rmse_mean", "id")], wide.loc[model, ("rmse_std", "id")])
        if model == best_id_model:
            id_txt = r"\textbf{" + id_txt + "}"

    if ("rmse_mean", "od") in wide.columns:
        od_txt = _pm(wide.loc[model, ("rmse_mean", "od")], wide.loc[model, ("rmse_std", "od")])
        if model == best_od_model:
            od_txt = r"\textbf{" + od_txt + "}"

    lines.append(f"{model} & {id_txt} & {od_txt} \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

latex_table = "\n".join(lines)
print(latex_table)

# Optional: save
latex_path = Path("outputs/advection_cno/cno_results_table.tex")
latex_path.parent.mkdir(parents=True, exist_ok=True)
latex_path.write_text(latex_table, encoding="utf-8")
print(f"Saved LaTeX table: {latex_path}")